# Physician Drug Adoption Prediction — Final ML 102B Notebook

**Business objective:** Predict which currently non-adopting physicians are most likely to prescribe the drug for the first time in the next quarter, and ultimately score the ~1,502 physicians in `Test_physicians.csv` for Quarter 11.

### Final methodology
- Merge `Input_data_file1.csv` (quarterly physician activity) with `Input_data_file2.csv` (static physician profile).
- Define the target as **next-quarter first adoption**.
- Keep only physicians who have **not adopted yet at the prediction point**.
- Create **Lag-1 and Lag-2 only** from historical quarterly activity.
- Add sensible 2-quarter derived features: average and change.
- Use correlation and VIF as **diagnostics**, not automatic deletion rules.
- Split the labeled historical observations chronologically into **80% development + 20% untouched historical test**.
- Within the first 80%, use rolling/forward temporal cross-validation.
- Compare Logistic Regression, Random Forest and XGBoost.
- Select the model primarily by **Lift@20%**, because sales capacity is limited.
- Evaluate once on the untouched historical 20%.
- Retrain the selected model on all labeled history available through Q10 and predict Q11 for the 1,502 test physicians.

The assignment screenshots specify the three data files and the data dictionary; the column names used below follow the supplied data dictionary/helper material. fileciteturn9file0L19-L51


In [ ]:
# CELL 2 — IMPORT LIBRARIES
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    confusion_matrix
)
from sklearn.inspection import permutation_importance

from statsmodels.stats.outliers_influence import variance_inflation_factor

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

print("Libraries imported successfully.")
print("XGBoost available:", XGBOOST_AVAILABLE)


## CELL 3 — Load the data

All three input files are assumed to be in the **same directory as this notebook**, so no directory/path configuration is required.


In [ ]:
# CELL 3 — LOAD INPUT DATA
file1 = pd.read_csv("Input_data_file1.csv")
file2 = pd.read_csv("Input_data_file2.csv")
test_physicians = pd.read_csv("Test_physicians.csv")

print("Input 1 shape:", file1.shape)
print("Input 2 shape:", file2.shape)
print("Test physicians shape:", test_physicians.shape)


In [ ]:
# CELL 4 — QUICK DATA INSPECTION
print("Input 1 columns:")
print(file1.columns.tolist())

print("\nInput 2 columns:")
print(file2.columns.tolist())

print("\nTest physician columns:")
print(test_physicians.columns.tolist())

display(file1.head())
display(file2.head())
display(test_physicians.head())


In [ ]:
# CELL 5 — COLUMN DEFINITIONS
TARGET_COL = "brand_prescribed"
ID_COL = "physician_id"
QTR_COL = "year_quarter"

# Quarterly behavioral/activity variables from Input_data_file1.csv
LAG_NUMERIC = [
    "total_representative_visits",
    "total_sample_dropped",
    "saving_cards_dropped",
    "vouchers_dropped",
    "total_seminar_as_attendee",
    "total_seminar_as_speaker",
    "total_prescriptions_for_indication1",
    "total_prescriptions_for_indication2",
    "total_prescriptions_for_indication3",
    "total_patient_with_commercial_insurance_plan",
    "total_patient_with_medicare_insurance_plan",
    "total_patient_with_medicaid_insurance_plan",
    "brand_web_impressions",
    "brand_ehr_impressions",
    "brand_enews_impressions",
    "brand_mobile_impressions",
    "brand_organic_web_visits",
    "brand_paidsearch_visits",
    "total_competitor_prescription",
    "new_prescriptions",
]

# Static physician profile variables from Input_data_file2.csv
CATEGORICAL_COLS = [
    "physician_hospital_affiliation",
    "physician_in_group_practice",
    "physician_gender",
    "physician_speciality",
    "physician_value_tier",
]

STATIC_NUM_COLS = [
    "urban",
    "percent",
    "physician_age",
    "physician_years_experience",
]

print("Quarterly numeric variables:", len(LAG_NUMERIC))
print("Categorical variables:", len(CATEGORICAL_COLS))
print("Static numeric variables:", len(STATIC_NUM_COLS))


In [ ]:
# CELL 6 — VALIDATE REQUIRED COLUMNS
required_input1 = {ID_COL, QTR_COL, TARGET_COL, *LAG_NUMERIC}
required_input2 = {ID_COL, *CATEGORICAL_COLS, *STATIC_NUM_COLS}

missing_input1 = sorted(required_input1 - set(file1.columns))
missing_input2 = sorted(required_input2 - set(file2.columns))
missing_test = sorted({ID_COL} - set(test_physicians.columns))

if missing_input1:
    raise ValueError(f"Missing Input 1 columns: {missing_input1}")
if missing_input2:
    raise ValueError(f"Missing Input 2 columns: {missing_input2}")
if missing_test:
    raise ValueError(f"Missing Test physicians columns: {missing_test}")

print("Column validation passed.")


In [ ]:
# CELL 7 — BASIC DATA QUALITY CHECKS
print("Input 1 duplicate rows:", file1.duplicated().sum())
print("Input 2 duplicate rows:", file2.duplicated().sum())
print("Input 1 duplicate physician-quarter rows:",
      file1.duplicated([ID_COL, QTR_COL]).sum())
print("Input 2 duplicate physician IDs:", file2[ID_COL].duplicated().sum())
print("Test duplicate physician IDs:", test_physicians[ID_COL].duplicated().sum())

print("\nUnique physicians:")
print("Input 1:", file1[ID_COL].nunique())
print("Input 2:", file2[ID_COL].nunique())
print("Test:", test_physicians[ID_COL].nunique())


In [ ]:
# CELL 8 — MERGE INPUT 1 + INPUT 2
# Input 1 is physician-quarter level; Input 2 should have one static profile per physician.

if file2[ID_COL].duplicated().any():
    raise ValueError("Input_data_file2.csv must contain one profile row per physician.")

rows_before = len(file1)

merged = file1.merge(
    file2,
    on=ID_COL,
    how="left",
    validate="many_to_one"
)

print("Rows before merge:", rows_before)
print("Rows after merge :", len(merged))
print("Merged shape     :", merged.shape)

if len(merged) != rows_before:
    raise ValueError("Merge changed the number of Input 1 rows.")

if merged.duplicated([ID_COL, QTR_COL]).any():
    raise ValueError("Merge created duplicate physician-quarter rows.")

print("Merge validation passed.")


In [ ]:
# CELL 9 — MISSING-VALUE ANALYSIS
missing_df = pd.DataFrame({
    "missing_count": merged.isna().sum(),
    "missing_pct": (merged.isna().mean() * 100).round(2)
}).sort_values("missing_pct", ascending=False)

display(missing_df)

print("Features with >=20% missingness:")
display(missing_df[missing_df["missing_pct"] >= 20])


### Missing-value rule

The 20–30% level is a **screening threshold**, not an automatic deletion rule. A high-missingness variable should only be removed when there is a reasonable business/data-quality justification. For the model pipeline, numeric variables are median-imputed and categorical variables are filled with the most frequent category, so preprocessing does not use information from validation/test folds. This follows the intended methodology in the supplied project specification. fileciteturn8file7L1132-L1149


In [ ]:
# CELL 10 — TARGET DISTRIBUTION / EDA
print("Current-quarter adoption rate:", f"{merged[TARGET_COL].mean():.2%}")
print("\nBrand prescribed counts:")
print(merged[TARGET_COL].value_counts(dropna=False))

adoption_trend = merged.groupby(QTR_COL)[TARGET_COL].agg(["mean", "count"])
adoption_trend.columns = ["adoption_rate", "n_physicians"]

display(adoption_trend)

plt.figure(figsize=(10, 4))
plt.plot(adoption_trend.index.astype(str), adoption_trend["adoption_rate"], marker="o")
plt.xticks(rotation=45)
plt.ylabel("Adoption rate")
plt.xlabel("Quarter")
plt.title("Current-quarter Drug Adoption Rate")
plt.tight_layout()
plt.show()


In [ ]:
# CELL 11 — NUMERICAL EDA
eda_numeric = [
    "total_representative_visits",
    "total_sample_dropped",
    "new_prescriptions",
    "total_competitor_prescription"
]

display(merged[eda_numeric].describe().round(2))
display(
    merged.groupby(TARGET_COL)[eda_numeric]
    .median()
    .round(2)
)


In [ ]:
# CELL 12 — CATEGORICAL EDA
for col in CATEGORICAL_COLS:
    print(f"\n--- {col} ---")
    display(merged[col].value_counts(dropna=False).head(15))


In [ ]:
# CELL 13 — CREATE PROPER QUARTER TIME INDEX
# year_quarter is supplied as YYYYQ, e.g. 201502 = Q2 of 2015.

merged[QTR_COL] = merged[QTR_COL].astype(str).str.zfill(6)

merged["year"] = merged[QTR_COL].str[:4].astype(int)
merged["quarter_number"] = merged[QTR_COL].str[4:6].astype(int)

if not merged["quarter_number"].isin([1, 2, 3, 4]).all():
    bad = sorted(merged.loc[~merged["quarter_number"].isin([1,2,3,4]), QTR_COL].unique())
    raise ValueError(f"Invalid quarter values found: {bad}")

# Consecutive quarters differ by exactly 1.
merged["time_index"] = merged["year"] * 4 + merged["quarter_number"]

merged = merged.sort_values([ID_COL, "time_index"]).reset_index(drop=True)

unique_times = sorted(merged["time_index"].unique())
print("Unique time indexes:", unique_times)
print("Number of quarters:", len(unique_times))


In [ ]:
# CELL 14 — CHECK PHYSICIAN QUARTER CONTINUITY
def continuity_check(df):
    gaps = []
    for physician, g in df.groupby(ID_COL):
        times = sorted(g["time_index"].unique())
        if len(times) > 1:
            diffs = np.diff(times)
            if not np.all(diffs == 1):
                gaps.append((physician, times))
    return gaps

continuity_gaps = continuity_check(merged)

print("Physicians with quarter gaps:", len(continuity_gaps))

if continuity_gaps:
    print("First few examples:")
    for item in continuity_gaps[:5]:
        print(item)
else:
    print("All physician histories are quarter-continuous.")


## Target definition

For a physician who has **not adopted yet** in quarter *t*:

**Information available through t → predict whether first adoption occurs in t+1.**

The next-quarter target is created only when the physician actually has a consecutive next quarter; we do not blindly use `shift(-1)` across missing quarters. fileciteturn8file1L225-L253


In [ ]:
# CELL 15 — CREATE NEXT-QUARTER TARGET
next_time = merged.groupby(ID_COL)["time_index"].shift(-1)
next_target = merged.groupby(ID_COL)[TARGET_COL].shift(-1)

merged["valid_next_qtr"] = (
    next_time == merged["time_index"] + 1
).astype(int)

merged["target_next_qtr"] = np.where(
    merged["valid_next_qtr"] == 1,
    next_target,
    np.nan
)

print("Rows with valid next quarter:", merged["valid_next_qtr"].sum())
print("Next-quarter adopters:", (merged["target_next_qtr"] == 1).sum())
print("Next-quarter non-adopters:", (merged["target_next_qtr"] == 0).sum())


## Current non-adopter logic

We are predicting **new adoption**, not continued prescribing.

- Before first adoption → eligible prediction observation.
- First adoption quarter → not a prediction source.
- After first adoption → no longer eligible as a new adopter.

This is the intended business definition in the supplied methodology. fileciteturn8file1L182-L214


In [ ]:
# CELL 16 — KEEP ONLY CURRENT NON-ADOPTERS
# IMPORTANT: this calculation is performed on the FULL merged panel
# before filtering rows, so adoption history is not lost.

previous_adoption = (
    merged.groupby(ID_COL)[TARGET_COL]
    .cummax()
    .groupby(merged[ID_COL])
    .shift(1, fill_value=0)
)

merged["adopted_before"] = previous_adoption.astype(int)

merged["adopted_first_qtr"] = (
    (merged[TARGET_COL] == 1) &
    (merged["adopted_before"] == 0)
).astype(int)

merged["is_non_adopter_now"] = (
    (merged["adopted_before"] == 0) &
    (merged[TARGET_COL] == 0)
).astype(int)

print("Current non-adopter rows:", merged["is_non_adopter_now"].sum())
print("First-adoption rows:", merged["adopted_first_qtr"].sum())


In [ ]:
# CELL 17 — VERIFY TARGET CONSTRUCTION
target_rows = merged[
    (merged["is_non_adopter_now"] == 1) &
    merged["target_next_qtr"].notna()
].copy()

lookup = merged.set_index([ID_COL, "time_index"])[TARGET_COL]

mismatch_count = 0

for _, row in target_rows.iterrows():
    actual = lookup.get(
        (row[ID_COL], int(row["time_index"]) + 1),
        np.nan
    )
    if pd.notna(actual) and int(actual) != int(row["target_next_qtr"]):
        mismatch_count += 1

print("Target mismatches:", mismatch_count)
assert mismatch_count == 0

print("Target construction verified.")


## Lag feature strategy

We use **Lag-1 and Lag-2 only**:

- Lag-1 = previous quarter.
- Lag-2 = two quarters ago.

For a Q11 prediction:
- Lag-1 = Q10
- Lag-2 = Q9

Lags are created on the **full chronological physician panel first**, and only then are current non-adopter rows selected. This prevents the filtering step from accidentally making an older quarter look like the immediately previous quarter. fileciteturn8file1L297-L340


In [ ]:
# CELL 18 — CREATE LAG-1 AND LAG-2 ON FULL PANEL
merged = merged.sort_values([ID_COL, "time_index"]).reset_index(drop=True)

for col in LAG_NUMERIC:
    merged[f"{col}_lag1"] = merged.groupby(ID_COL)[col].shift(1)
    merged[f"{col}_lag2"] = merged.groupby(ID_COL)[col].shift(2)

lag1_cols = [f"{c}_lag1" for c in LAG_NUMERIC]
lag2_cols = [f"{c}_lag2" for c in LAG_NUMERIC]

print("Lag-1 features:", len(lag1_cols))
print("Lag-2 features:", len(lag2_cols))


In [ ]:
# CELL 19 — VALIDATE LAG CONTINUITY
# A lag is considered valid only when the historical quarter is actually t-1/t-2.

for col in LAG_NUMERIC:
    l1_valid = merged.groupby(ID_COL)["time_index"].shift(1) == merged["time_index"] - 1
    l2_valid = merged.groupby(ID_COL)["time_index"].shift(2) == merged["time_index"] - 2

    merged.loc[~l1_valid, f"{col}_lag1"] = np.nan
    merged.loc[~l2_valid, f"{col}_lag2"] = np.nan

print("Lag continuity validation applied.")


In [ ]:
# CELL 20 — BUILD MODELING DATA
model_data = merged[
    merged["is_non_adopter_now"] == 1
].copy()

# We can only train/evaluate when the next-quarter outcome is known.
labeled_data = model_data[
    model_data["target_next_qtr"].notna()
].copy()

labeled_data["target_next_qtr"] = labeled_data["target_next_qtr"].astype(int)

print("Eligible non-adopter rows:", len(model_data))
print("Labeled modeling rows:", len(labeled_data))
print("Unique physicians:", labeled_data[ID_COL].nunique())
print("Target adoption rate:", f"{labeled_data['target_next_qtr'].mean():.2%}")


## Derived features

From each Lag-1/Lag-2 pair we create:
- `avg_2q` = average activity across the last two quarters.
- `change` = Lag-1 − Lag-2, representing recent momentum.

We deliberately do **not** create a 2-quarter sum because it is mathematically redundant with the average (`sum = 2 × average`). No Lag-3/Lag-4 features are introduced. fileciteturn8file1L343-L348


In [ ]:
# CELL 21 — DERIVED LAG FEATURES
for col in LAG_NUMERIC:
    l1 = f"{col}_lag1"
    l2 = f"{col}_lag2"

    labeled_data[f"{col}_avg_2q"] = (
        labeled_data[l1] + labeled_data[l2]
    ) / 2.0

    labeled_data[f"{col}_change"] = (
        labeled_data[l1] - labeled_data[l2]
    )

avg_cols = [f"{c}_avg_2q" for c in LAG_NUMERIC]
change_cols = [f"{c}_change" for c in LAG_NUMERIC]

print("Average features:", len(avg_cols))
print("Change features:", len(change_cols))


In [ ]:
# CELL 22 — DEFINE FINAL FEATURES
NUMERIC_FEATURES = (
    lag1_cols
    + lag2_cols
    + avg_cols
    + change_cols
    + STATIC_NUM_COLS
)

CAT_FEATURES = CATEGORICAL_COLS.copy()

FEATURES = NUMERIC_FEATURES + CAT_FEATURES

# Safety checks
assert TARGET_COL not in FEATURES
assert "target_next_qtr" not in FEATURES
assert ID_COL not in FEATURES
assert QTR_COL not in FEATURES
assert all("_lag3" not in c and "_lag4" not in c for c in FEATURES)

print("Numeric model features:", len(NUMERIC_FEATURES))
print("Categorical model features:", len(CAT_FEATURES))
print("Total model features before one-hot encoding:", len(FEATURES))


## Feature diagnostics

Correlation and VIF are used as **diagnostics**. We do not automatically delete a feature just because it is correlated or has high VIF.

This is especially important because:
- tree-based models can handle correlated predictors reasonably well;
- derived Lag-1/Lag-2 features are naturally correlated;
- automatic deletion can remove useful business signal.

The final model interpretation will additionally use permutation/native feature importance.


In [ ]:
# CELL 23 — CORRELATION DIAGNOSTIC
corr_features = [c for c in NUMERIC_FEATURES if c in labeled_data.columns]

corr_matrix = labeled_data[corr_features + ["target_next_qtr"]].corr(
    method="spearman"
)

target_corr = (
    corr_matrix["target_next_qtr"]
    .drop("target_next_qtr")
    .sort_values(key=np.abs, ascending=False)
)

display(target_corr.head(25).to_frame("spearman_correlation_with_target"))


In [ ]:
# CELL 24 — HIGH-CORRELATION PAIR DIAGNOSTIC
corr_abs = labeled_data[corr_features].corr(method="spearman").abs()

upper = corr_abs.where(
    np.triu(np.ones(corr_abs.shape), k=1).astype(bool)
)

high_corr_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={
        "level_0": "feature_1",
        "level_1": "feature_2",
        0: "abs_spearman_corr"
    })
    .sort_values("abs_spearman_corr", ascending=False)
)

display(high_corr_pairs.head(30))


In [ ]:
# CELL 25 — VIF DIAGNOSTIC
# VIF is mainly useful for diagnosing multicollinearity in linear-model inputs.
# We calculate it on the development-period numeric features only.

LABELED_QTRS = sorted(labeled_data["time_index"].unique())

# Use the development period defined below for VIF, before fitting the final models.
n_test_qtrs = max(1, int(np.ceil(len(LABELED_QTRS) * 0.20)))
DEV_QTRS = LABELED_QTRS[:-n_test_qtrs]
TEST_QTRS = LABELED_QTRS[-n_test_qtrs:]

dev_preview = labeled_data[
    labeled_data["time_index"].isin(DEV_QTRS)
].copy()

vif_features = [c for c in NUMERIC_FEATURES if c in dev_preview.columns]

vif_sample = dev_preview[vif_features].sample(
    n=min(10000, len(dev_preview)),
    random_state=42
)

vif_sample = pd.DataFrame(
    SimpleImputer(strategy="median").fit_transform(vif_sample),
    columns=vif_features
)

# Drop zero-variance columns before VIF.
non_constant = vif_sample.columns[vif_sample.nunique() > 1].tolist()
vif_sample = vif_sample[non_constant]

vif_results = []

for i, col in enumerate(vif_sample.columns):
    try:
        value = variance_inflation_factor(vif_sample.values, i)
    except Exception:
        value = np.inf

    vif_results.append({
        "feature": col,
        "VIF": value
    })

vif_df = pd.DataFrame(vif_results).sort_values(
    "VIF", ascending=False
)

display(vif_df.head(30))


## 80% development / 20% historical test

The split is **chronological**, not random.

- First ~80% of labeled source quarters → development data.
- Latest ~20% → untouched historical test data.
- Temporal cross-validation happens **only inside the 80% development period**.
- The final 20% is not used for feature selection or model selection.

This matches the required longitudinal-data strategy. fileciteturn8file6L976-L1019


In [ ]:
# CELL 26 — FINAL CHRONOLOGICAL 80/20 SPLIT
DEV_QTRS = LABELED_QTRS[:-n_test_qtrs]
TEST_QTRS = LABELED_QTRS[-n_test_qtrs:]

dev = labeled_data[
    labeled_data["time_index"].isin(DEV_QTRS)
].copy()

historical_test = labeled_data[
    labeled_data["time_index"].isin(TEST_QTRS)
].copy()

assert set(DEV_QTRS).isdisjoint(TEST_QTRS)
assert max(DEV_QTRS) < min(TEST_QTRS)

print("Development quarters:", DEV_QTRS)
print("Historical test quarters:", TEST_QTRS)
print("Development rows:", len(dev))
print("Historical test rows:", len(historical_test))
print("Development adoption rate:", f"{dev['target_next_qtr'].mean():.2%}")
print("Historical test adoption rate:", f"{historical_test['target_next_qtr'].mean():.2%}")


In [ ]:
# CELL 27 — TEMPORAL CROSS-VALIDATION FOLDS
def create_temporal_folds(df, time_col="time_index"):
    quarters = sorted(df[time_col].unique())

    # Each validation quarter must have at least two earlier quarters available for training.
    folds = []
    for i in range(2, len(quarters)):
        train_q = quarters[:i]
        valid_q = [quarters[i]]
        folds.append((train_q, valid_q))

    return folds

folds = create_temporal_folds(dev)

for i, (train_q, valid_q) in enumerate(folds, 1):
    print(
        f"Fold {i}: "
        f"Train {min(train_q)} -> {max(train_q)}, "
        f"Validation = {valid_q}"
    )


### Why temporal CV?

This is panel/time-series-like physician data. We want the model to learn:

**past physician behavior → future physician adoption**

Randomly mixing rows could allow later behavior to influence an earlier validation period. Temporal folds prevent that. fileciteturn8file0L14-L50


In [ ]:
# CELL 28 — PREPROCESSING PIPELINE
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

def make_preprocessor():
    return ColumnTransformer([
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CAT_FEATURES)
    ])

print("Numeric: median imputation -> scaling")
print("Categorical: most-frequent imputation -> one-hot encoding")


In [ ]:
# CELL 29 — DEFINE MODELS
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        C=1.0,
        solver="lbfgs"
    ),

    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

if XGBOOST_AVAILABLE:
    models["XGBoost"] = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

print("Models to evaluate:", list(models.keys()))


In [ ]:
# CELL 30 — EVALUATION FUNCTIONS
def lift_at_20(y_true, y_probability):
    y_true = np.asarray(y_true)
    y_probability = np.asarray(y_probability)

    n_top = max(1, int(np.ceil(len(y_true) * 0.20)))
    ranking = np.argsort(y_probability)[::-1]
    top_indices = ranking[:n_top]

    overall_rate = y_true.mean()
    top_rate = y_true[top_indices].mean()

    if overall_rate == 0:
        return np.nan

    return top_rate / overall_rate


def evaluate_predictions(y_true, probabilities, threshold=0.50):
    predictions = (np.asarray(probabilities) >= threshold).astype(int)

    return {
        "ROC-AUC": roc_auc_score(y_true, probabilities),
        "PR-AUC": average_precision_score(y_true, probabilities),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "F1": f1_score(y_true, predictions, zero_division=0),
        "Lift@20%": lift_at_20(y_true, probabilities)
    }


In [ ]:
# CELL 31 — TEMPORAL CROSS-VALIDATION FOR ALL MODELS
cv_results = []

for model_name, base_model in models.items():

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    for fold_number, (train_quarters, validation_quarters) in enumerate(folds, 1):

        train_data = dev[
            dev["time_index"].isin(train_quarters)
        ].copy()

        validation_data = dev[
            dev["time_index"].isin(validation_quarters)
        ].copy()

        X_train = train_data[FEATURES]
        y_train = train_data["target_next_qtr"].astype(int)

        X_valid = validation_data[FEATURES]
        y_valid = validation_data["target_next_qtr"].astype(int)

        # New preprocessing + model pipeline for every fold.
        # Therefore imputation/scaling/encoding are fitted only on that fold's training data.
        pipeline = Pipeline([
            ("preprocessor", make_preprocessor()),
            ("model", base_model)
        ])

        pipeline.fit(X_train, y_train)

        probabilities = pipeline.predict_proba(X_valid)[:, 1]

        metrics = evaluate_predictions(y_valid, probabilities)

        metrics.update({
            "model": model_name,
            "fold": fold_number,
            "train_start": min(train_quarters),
            "train_end": max(train_quarters),
            "validation_quarter": validation_quarters[0]
        })

        cv_results.append(metrics)

cv_results_df = pd.DataFrame(cv_results)

display(cv_results_df.round(4))


In [ ]:
# CELL 32 — MODEL COMPARISON
model_comparison = (
    cv_results_df
    .groupby("model")
    .agg(
        mean_ROC_AUC=("ROC-AUC", "mean"),
        std_ROC_AUC=("ROC-AUC", "std"),
        mean_PR_AUC=("PR-AUC", "mean"),
        std_PR_AUC=("PR-AUC", "std"),
        mean_Precision=("Precision", "mean"),
        mean_Recall=("Recall", "mean"),
        mean_F1=("F1", "mean"),
        mean_Lift20=("Lift@20%", "mean"),
        std_Lift20=("Lift@20%", "std")
    )
    .reset_index()
    .sort_values(
        ["mean_Lift20", "mean_PR_AUC"],
        ascending=False
    )
)

display(model_comparison.round(4))


## Model selection

The primary selection metric is **Lift@20%**. If two models are close on Lift@20%, PR-AUC and ROC-AUC can be used as secondary considerations.

Lift@20% answers the business question:

> If sales representatives can target only the top 20% of physicians ranked by predicted probability, how much better is that targeting than selecting physicians randomly?

The project specification explicitly identifies Lift@20% as the primary business metric. fileciteturn8file0L109-L142


In [ ]:
# CELL 33 — SELECT BEST MODEL
BEST_MODEL_NAME = model_comparison.iloc[0]["model"]

print("Selected model:", BEST_MODEL_NAME)
print("Primary selection criterion: mean Lift@20%")


In [ ]:
# CELL 34 — FIT SELECTED MODEL ON COMPLETE 80% DEVELOPMENT DATA
best_model_pipeline = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", models[BEST_MODEL_NAME])
])

X_dev = dev[FEATURES]
y_dev = dev["target_next_qtr"].astype(int)

best_model_pipeline.fit(X_dev, y_dev)

print("Selected model fitted on the full 80% development data.")


In [ ]:
# CELL 35 — FINAL UNTOUCHED HISTORICAL 20% TEST
X_test = historical_test[FEATURES]
y_test = historical_test["target_next_qtr"].astype(int)

test_probability = best_model_pipeline.predict_proba(X_test)[:, 1]

final_test_metrics = evaluate_predictions(
    y_test,
    test_probability
)

print("=" * 70)
print("FINAL HISTORICAL TEST PERFORMANCE")
print("=" * 70)

for metric, value in final_test_metrics.items():
    print(f"{metric:15s}: {value:.4f}")


In [ ]:
# CELL 36 — HISTORICAL TEST CONFUSION MATRIX
test_prediction = (test_probability >= 0.50).astype(int)

cm = confusion_matrix(y_test, test_prediction)

display(pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
))

print("Threshold 0.50 is used only for Precision/Recall/F1.")
print("Business targeting is ranking-based and uses the top 20%.")


## Feature importance

The assignment asks us to identify the features most relevant for prediction. We report both:
- native model importance where available; and
- permutation importance using average precision.

The feature-importance analysis is performed on the development data, not the untouched final test set.


In [ ]:
# CELL 37 — FEATURE IMPORTANCE
feature_names = (
    best_model_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

fitted_model = best_model_pipeline.named_steps["model"]

if hasattr(fitted_model, "feature_importances_"):
    native_importance = fitted_model.feature_importances_
elif hasattr(fitted_model, "coef_"):
    native_importance = np.abs(fitted_model.coef_[0])
else:
    native_importance = np.full(len(feature_names), np.nan)

perm = permutation_importance(
    best_model_pipeline,
    X_dev,
    y_dev,
    scoring="average_precision",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "native_importance": native_importance,
    "permutation_importance": perm.importances_mean
}).sort_values(
    "permutation_importance",
    ascending=False
)

display(feature_importance_df.head(25).round(5))


In [ ]:
# CELL 38 — RETRAIN FINAL MODEL ON ALL LABELED HISTORY THROUGH Q10
# At this point model selection and historical testing are complete.
# We can use all labeled historical source quarters available before Q11.

all_labeled_history = labeled_data.copy()

final_model_pipeline = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", models[BEST_MODEL_NAME])
])

X_all = all_labeled_history[FEATURES]
y_all = all_labeled_history["target_next_qtr"].astype(int)

final_model_pipeline.fit(X_all, y_all)

print("Final model trained on all labeled historical observations.")
print("Q11 outcome is not used because it is unknown.")


# Quarter 11 prediction

`Test_physicians.csv` contains the ~1,502 physicians who never prescribed the drug during the historical period and for whom Q11 adoption must be predicted.

For Q11:
- **Lag-1 = Q10**
- **Lag-2 = Q9**

Only information available through Q10 is used. fileciteturn8file2L359-L383


In [ ]:
# CELL 39 — LOAD / VALIDATE Q11 PHYSICIAN POPULATION
q11_physicians = test_physicians[[ID_COL]].drop_duplicates().copy()

print("Q11 physicians:", len(q11_physicians))

if len(q11_physicians) != 1502:
    print("Warning: expected approximately 1,502 physicians; actual count is", len(q11_physicians))


In [ ]:
# CELL 40 — VERIFY Q11 PHYSICIANS NEVER ADOPTED HISTORICALLY
q11_history = merged[
    merged[ID_COL].isin(q11_physicians[ID_COL])
].copy()

historical_adoption = q11_history.groupby(ID_COL)[TARGET_COL].max()

previous_adopters = historical_adoption[
    historical_adoption == 1
]

print("Q11 physicians with previous adoption:", len(previous_adopters))

if len(previous_adopters) > 0:
    print("Warning: some test physicians have historical adoption.")
    print("These IDs should be reviewed against the supplied Test_physicians definition.")


In [ ]:
# CELL 41 — BUILD Q11 LAG-1 AND LAG-2
# Identify the latest two historical quarters in the modeling data.
all_times = sorted(merged["time_index"].unique())

Q10_INDEX = all_times[-1]
Q9_INDEX = all_times[-2]

q10_data = merged[
    merged["time_index"] == Q10_INDEX
].copy()

q9_data = merged[
    merged["time_index"] == Q9_INDEX
].copy()

q11_features = q11_physicians.copy()

q10_selected = q10_data[
    [ID_COL] + LAG_NUMERIC
].rename(
    columns={c: f"{c}_lag1" for c in LAG_NUMERIC}
)

q9_selected = q9_data[
    [ID_COL] + LAG_NUMERIC
].rename(
    columns={c: f"{c}_lag2" for c in LAG_NUMERIC}
)

q11_features = q11_features.merge(
    q10_selected,
    on=ID_COL,
    how="left",
    validate="one_to_one"
)

q11_features = q11_features.merge(
    q9_selected,
    on=ID_COL,
    how="left",
    validate="one_to_one"
)

print("Q11 Lag-1 = historical latest quarter (Q10)")
print("Q11 Lag-2 = historical previous quarter (Q9)")
print("Q10 rows found:", len(q10_data))
print("Q9 rows found:", len(q9_data))


In [ ]:
# CELL 42 — VALIDATE Q11 LAG VALUES
for col in LAG_NUMERIC:

    expected_lag1 = q10_data[[ID_COL, col]].rename(
        columns={col: "expected"}
    )

    check1 = q11_features[
        [ID_COL, f"{col}_lag1"]
    ].merge(expected_lag1, on=ID_COL, how="left")

    assert np.allclose(
        check1[f"{col}_lag1"],
        check1["expected"],
        equal_nan=True
    )

    expected_lag2 = q9_data[[ID_COL, col]].rename(
        columns={col: "expected"}
    )

    check2 = q11_features[
        [ID_COL, f"{col}_lag2"]
    ].merge(expected_lag2, on=ID_COL, how="left")

    assert np.allclose(
        check2[f"{col}_lag2"],
        check2["expected"],
        equal_nan=True
    )

print("Q11 Lag-1/Lag-2 validation passed.")


In [ ]:
# CELL 43 — CREATE Q11 DERIVED FEATURES
for col in LAG_NUMERIC:
    l1 = f"{col}_lag1"
    l2 = f"{col}_lag2"

    q11_features[f"{col}_avg_2q"] = (
        q11_features[l1] + q11_features[l2]
    ) / 2.0

    q11_features[f"{col}_change"] = (
        q11_features[l1] - q11_features[l2]
    )

print("Q11 derived features created.")


In [ ]:
# CELL 44 — ADD STATIC PHYSICIAN FEATURES
static_profile = file2[
    [ID_COL] + STATIC_NUM_COLS + CAT_FEATURES
].drop_duplicates(ID_COL)

q11_features = q11_features.merge(
    static_profile,
    on=ID_COL,
    how="left",
    validate="one_to_one"
)

print("Q11 feature rows:", len(q11_features))

missing_q11_features = sorted(
    set(FEATURES) - set(q11_features.columns)
)

if missing_q11_features:
    raise ValueError(
        f"Q11 is missing model features: {missing_q11_features}"
    )

print("Q11 feature compatibility passed.")


In [ ]:
# CELL 45 — GENERATE Q11 ADOPTION PROBABILITIES
q11_probability = final_model_pipeline.predict_proba(
    q11_features[FEATURES]
)[:, 1]

predictions = pd.DataFrame({
    ID_COL: q11_features[ID_COL],
    "predicted_adoption_probability": q11_probability
})

print("Physicians scored:", len(predictions))
display(predictions.head())


In [ ]:
# CELL 46 — RANK PHYSICIANS
predictions = predictions.sort_values(
    "predicted_adoption_probability",
    ascending=False
).reset_index(drop=True)

predictions["rank"] = np.arange(1, len(predictions) + 1)

display(predictions.head(20))


In [ ]:
# CELL 47 — CREATE TARGETING SEGMENTS
n_physicians = len(predictions)

high_cutoff = int(np.ceil(0.20 * n_physicians))
medium_cutoff = int(np.ceil(0.50 * n_physicians))

predictions["target_group"] = np.select(
    [
        predictions["rank"] <= high_cutoff,
        predictions["rank"] <= medium_cutoff
    ],
    [
        "High Priority",
        "Medium Priority"
    ],
    default="Low Priority"
)

print("High Priority:", high_cutoff)
print("Medium Priority:", medium_cutoff - high_cutoff)
print("Low Priority:", n_physicians - medium_cutoff)

display(predictions.head(20))


## Business interpretation of Lift@20%

If the model's Lift@20% is, for example, **2.5**, then the top 20% of ranked physicians contain adopters at **2.5 times the overall adoption rate**.

This is why Lift@20% is more directly aligned to the sales use case than plain accuracy. fileciteturn8file0L128-L142


In [ ]:
# CELL 48 — EXPORT REQUIRED OUTPUTS
predictions.to_csv(
    "physician_adoption_predictions.csv",
    index=False
)

model_comparison.to_csv(
    "model_comparison.csv",
    index=False
)

cv_results_df.to_csv(
    "validation_results.csv",
    index=False
)

feature_importance_df.to_csv(
    "feature_importance.csv",
    index=False
)

pd.DataFrame([final_test_metrics]).to_csv(
    "final_historical_test_metrics.csv",
    index=False
)

vif_df.to_csv(
    "development_vif.csv",
    index=False
)

print("Output files created in the same directory as the notebook:")
print("1. physician_adoption_predictions.csv")
print("2. model_comparison.csv")
print("3. validation_results.csv")
print("4. feature_importance.csv")
print("5. final_historical_test_metrics.csv")
print("6. development_vif.csv")


In [ ]:
# CELL 49 — FINAL METHODOLOGY CHECKLIST
checks = {
    "Input 1 and Input 2 merged correctly":
        len(merged) == len(file1),

    "No duplicate physician-quarter rows":
        merged.duplicated([ID_COL, QTR_COL]).sum() == 0,

    "Target is next-quarter adoption":
        labeled_data["target_next_qtr"].notna().all(),

    "Only current non-adopters modeled":
        model_data["is_non_adopter_now"].eq(1).all(),

    "Development/test quarters do not overlap":
        set(DEV_QTRS).isdisjoint(TEST_QTRS),

    "Development is earlier than test":
        max(DEV_QTRS) < min(TEST_QTRS),

    "No Lag-3/Lag-4":
        all("_lag3" not in c and "_lag4" not in c for c in FEATURES),

    "Target excluded from features":
        TARGET_COL not in FEATURES and "target_next_qtr" not in FEATURES,

    "Physician ID excluded from features":
        ID_COL not in FEATURES,

    "Temporal CV used only inside development":
        all(
            max(train_q) < min(valid_q)
            for train_q, valid_q in folds
        ),

    "Q11 physician IDs are scored":
        len(predictions) == len(q11_physicians),

    "Predictions sorted by probability":
        predictions["predicted_adoption_probability"].is_monotonic_decreasing
}

check_df = pd.DataFrame({
    "check": list(checks.keys()),
    "passed": list(checks.values())
})

display(check_df)

assert check_df["passed"].all()

print("ALL FINAL CHECKS PASSED.")


# Final project conclusion

The finalized workflow predicts **next-quarter first adoption** for currently non-adopting physicians using only information available before the prediction quarter.

The historical labeled data is split chronologically into an approximately **80% development period and 20% untouched historical test period**. Temporal cross-validation is performed only inside the development period. The selected model is then evaluated once on the historical test period before being retrained on all labeled history available through Q10.

For the actual business prediction, Q11 physicians are scored using:
- Q10 as Lag-1,
- Q9 as Lag-2,
- two-quarter average/change features,
- static physician characteristics.

The physicians are ranked by predicted adoption probability and the top 20% are identified as the highest-priority sales targets. This matches the supplied project requirement that the final output contain probability, rank and targeting group. fileciteturn8file2L359-L445
